# Lesson 03 - Agentic Design Patterns

In this lesson, we explore three foundational design patterns for building effective AI agents:

1. **Clear Agent Instructions** — Crafting precise, role-defining prompts that guide agent behavior
2. **Structured Output with Pydantic Models** — Ensuring agents return predictable, validated data
3. **Single Responsibility Agents** — Designing focused agents that each do one thing well

We'll apply each pattern to a **travel destination recommender** scenario, progressively building a system that can suggest destinations, check availability, and handle logistics.

## Setup

In [1]:
%pip install agent-framework azure-ai-projects agent-framework-foundry azure-identity pydantic --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel, Field

# from agent_framework.azure import AzureAIProjectAgentProvider
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

load_dotenv()

foundry_client = FoundryChatClient(
  credential=AzureCliCredential(),
  project_endpoint=os.getenv("FOUNDRY_PROJECT_ENDPOINT"),
  model=os.getenv("FOUNDRY_MODEL")
)


## Pattern 1: Clear Agent Instructions

The most impactful pattern is also the simplest: writing clear, detailed instructions for your agent.

Good instructions define:
- **Who** the agent is (persona and tone)
- **What** it should do (step-by-step responsibilities)
- **How** it should behave (constraints and style)

Below, we create a travel concierge agent with explicit instructions that shape every response it produces.

In [6]:
# agent = await provider.create_agent(
#     name="TravelConcierge",
#     instructions="""You are a luxury travel concierge named Alex. Your role is to:
# 1. Understand the traveler's preferences (budget, climate, activities)
# 2. Check destination availability before making recommendations
# 3. Provide detailed, personalized travel suggestions
# 4. Always mention visa requirements and best travel seasons
# Be warm, professional, and enthusiastic about travel.""",
# )

agent = Agent(
  client=foundry_client,
  instructions="""
    You are a luxury travel concierge named Alex. Your role is to:
      1. Understand the traveler's preferences (budget, climate, activities)
      2. Check destination availability before making recommendations
      3. Provide detailed, personalized travel suggestions
      4. Always mention visa requirements and best travel seasons
    Be warm, professional, and enthusiastic about travel.
    """,
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

Thank you for sharing your travel preferences! A week-long vacation centered on great food and rich history sounds absolutely wonderful. To tailor the perfect recommendations for you, could you please share a bit more about:

1. Your preferred climate—do you enjoy warm, temperate, or cooler destinations?
2. Are you interested in exploring a particular region or continent?
3. Any specific activities you enjoy besides food and history, like guided tours, museum visits, or cultural experiences?
4. Your departure city or country, so I can check the best travel options and visa requirements.

This will help me ensure I recommend destinations that are available and a perfect fit for your interests and budget. Looking forward to your reply!


## Pattern 2: Structured Output with Pydantic Models

Free-form text is useful for conversation, but downstream systems need structured data.
By pairing **Pydantic models** with a **tool function**, we can:

- Define an exact schema for the agent's output
- Validate responses automatically
- Integrate agent results into application logic reliably

We also introduce a tool that returns destination details so the agent grounds its recommendations in real data.

In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int

class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str

destination_detail = {
    "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
    "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$1000/week",
    "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
}

@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, Field(description="The destination to look up")]
) -> str:
    """Get details about a vacation destination."""
    return destination_detail.get(destination, f"{destination}: No information available.")

@tool(approval_mode="never_require")
def list_destinations() -> str:
    """List all about a vacation destination."""
    return str(destination_detail)

structured_agent = Agent(
    client=foundry_client,
    name="StructuredTravelExpert",
    instructions=(
        "You are a travel expert. Recommend exactly 3 destinations based on traveler preferences."
        "First, call list_available_destinations to see what's available. "
        "Always call get_destination_details for each destination before recommending it."
    ),
    tools=[get_destination_details, list_destinations]
)

response = await structured_agent.run(
    "Recommend destinations for a culture-loving traveler with a $2000 budget",
    options={"response_format": TravelRecommendations},
)

if response.value:
    travel_data: TravelRecommendations = response.value

    print(f"\n{travel_data.personalized_note}\n")
    for rec in travel_data.recommendations:
        status = "Available" if rec.available else "Not Available"
        print(f"{rec.destination} [{status}]")
        print(f"   Best season : {rec.best_season}")
        print(f"   Budget      : ~${rec.estimated_budget_usd:,}/week")
        print(f"   Highlights  : {', '.join(rec.highlights)}")
        print()
else:
    print("Could not parse structured response:", response.text)


For a culture-loving traveler with a $2000 budget, Barcelona is the best fit with its rich architecture and vibrant nightlife within your budget. Tokyo offers incredible cultural experiences but is above your budget. I recommend Barcelona as the top choice.

Barcelona [Available]
   Best season : May-Jun
   Budget      : ~$2,000/week
   Highlights  : Beach, Architecture, Nightlife

Tokyo [Available]
   Best season : Mar-Apr
   Budget      : ~$10,000/week
   Highlights  : Culture, Food, Technology



Giải thích đoạn code trên:

Hai class này là các mô hình dữ liệu sử dụng Pydantic để định nghĩa cấu trúc dữ liệu đầu ra cho agent, giúp đảm bảo dữ liệu trả về có định dạng nhất quán, dễ kiểm tra và sử dụng trong ứng dụng.

#### 1. `DestinationRecommendation`
Đây là class mô tả một gợi ý điểm đến du lịch cụ thể, gồm các thuộc tính:
- `destination` (str): Tên điểm đến (ví dụ: "Barcelona", "Tokyo").
- `available` (bool): Điểm đến này hiện có thể đi được không (True/False).
- `best_season` (str): Mùa du lịch lý tưởng nhất cho điểm đến này (ví dụ: "May-Jun").
- `highlights` (list[str]): Danh sách các điểm nổi bật hoặc hoạt động hấp dẫn tại điểm đến (ví dụ: ["beach", "architecture"]).
- `estimated_budget_usd` (int): Ngân sách ước tính (USD) cho chuyến đi đến điểm này.

Class này giúp agent trả về thông tin chi tiết, có cấu trúc về từng điểm đến.

#### 2. `TravelRecommendations`
Đây là class tổng hợp các gợi ý điểm đến, gồm:
- `recommendations` (list[DestinationRecommendation]): Danh sách các gợi ý điểm đến, mỗi phần tử là một instance của class `DestinationRecommendation` ở trên.
- `personalized_note` (str): Ghi chú cá nhân hóa dành cho người dùng, thường là lời khuyên hoặc nhận xét tổng quan dựa trên sở thích của người dùng.

Class này giúp agent trả về nhiều gợi ý cùng lúc, kèm theo lời nhắn riêng phù hợp với yêu cầu của người dùng.

#### **Lưu ý QUAN TRỌNG:**
- Nếu như câu prompt không chứa bất kỳ từ nào liên quan tới tham số `destination` của hàm `get_destination_details` thì sẽ không gọi được function calling này
- Vì vậy, để giải quyết có thể thêm 1 function calling không chứa tham số để khi prompt không chứa tham số có thể lấy chính xác địa điểm

## Pattern 3: Single Responsibility Agents

Complex tasks benefit from splitting work across multiple focused agents, each with a single responsibility:

- A **Destination Expert** that knows about places and availability
- A **Logistics Planner** that handles flights, hotels, and itineraries

This mirrors the software engineering principle of *separation of concerns* — each agent is easier to test, maintain, and improve independently.

In [27]:
# destination_agent = await provider.create_agent(
#     name="DestinationExpert",
#     tools=[get_destination_details],
#     instructions="""You are a destination research specialist. Your only job is to:
# 1. Evaluate destinations based on traveler preferences
# 2. Check availability using the provided tool
# 3. Return a short ranked list with pros/cons
# Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
# )

destination_agent = Agent(
    client=foundry_client,
    name="DestinationExpert",
    tools=[list_destinations],
    instructions="""
        You are a destination research specialist. Your only job is to:
        1. Evaluate destinations based on traveler preferences
        2. Check availability using the provided tool
        3. Return a short ranked list with pros/cons
        Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

# logistics_agent = await provider.create_agent(
#     name="LogisticsPlanner",
#     instructions="""You are a travel logistics planner. Your only job is to:
# 1. Create a day-by-day itinerary for the chosen destination
# 2. Suggest flight and hotel options within the stated budget
# 3. Note visa requirements and travel insurance recommendations
# Do NOT recommend destinations — another agent handles that.""",
# )

logistics_agent = Agent(
    client=foundry_client,
    name="LogisticsPlanner",
    instructions="""
        You are a travel logistics planner. Your only job is to:
        1. Create a day-by-day itinerary for the chosen destination
        2. Suggest flight and hotel options within the stated budget
        3. Note visa requirements and travel insurance recommendations
        Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
    "Pick the single best destination from the available options and commit to it."
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

=== Destination Expert ===
The best destination for a week of culture and food under $2500 is Barcelona. It offers rich culture with stunning architecture, vibrant nightlife, and great food experiences. The cost is around $2000 for a week, which fits your budget well.

Pros:
- Rich cultural experiences including art and history
- Great food scene with diverse culinary options
- Beaches for relaxation
- Fits under the $2500 budget

Cons:
- May be crowded during peak season (May-Jun)
- Nightlife and crowds might not suit all preferences

I recommend Barcelona as your destination.

=== Logistics Planner ===
Certainly! Here's a detailed week-long itinerary to explore culture and food in Barcelona within a $2500 budget:

---

### Barcelona 7-Day Itinerary: Culture & Food Experience

---

#### Day 1: Arrival & Gothic Quarter Introduction
- **Morning:** Arrive in Barcelona El Prat Airport (BCN), take Aerobus or taxi to your hotel.
- **Afternoon:** Check-in & Light lunch nearby.
- **Evening:**

## Summary

In this lesson we applied three agentic design patterns to a travel recommender scenario:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | Define persona, responsibilities, and constraints up front | Consistent, on-brand agent behavior |
| **Structured Output** | Use Pydantic models as the response format | Validated, machine-readable results |
| **Single Responsibility** | Give each agent one focused job | Easier to test, maintain, and compose |

These patterns compose naturally — you can combine clear instructions with structured output inside a single-responsibility agent to build robust, production-ready systems.